# Phase-programmable 3 × 3 decoder

Optimize a 2.88 × 2.88 µm design by averaging three single-port equal-splitting responses. The full 36 × 36 parameter array is independent. Default: 13 wavelengths from 1470 to 1530 nm, 60 Adam updates.

This notebook uses the shared, tested implementation in `src/coherent_multiport/`. Run cells in order after installing the package from the repository root. The initial preview is local. Cloud optimization and final validation have separate explicit switches.

## 1. Load the research configuration

Lengths in the simulation are in micrometres. Port order is top, middle, bottom. Use a new output directory for a fresh optimization; an existing checkpoint resumes automatically.

In [ ]:
from pathlib import Path
import logging

import matplotlib.pyplot as plt
import numpy as np

from coherent_multiport import load_config
from coherent_multiport.simulation import build_simulation, initial_parameters
from coherent_multiport.optimization import optimize
from coherent_multiport.postprocess import evaluate

ROOT = Path.cwd().resolve()
if not (ROOT / "configs").is_dir() and (ROOT.parent / "configs").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "configs").is_dir():
    raise FileNotFoundError("Open this notebook from the repository or notebooks directory")

cfg = load_config(ROOT / "configs" / "decoder.yaml")
OUTPUT = ROOT / "results" / "decoder"
logging.basicConfig(level=logging.INFO, format="%(message)s")
print(f"Design: {cfg.design_x:.2f} × {cfg.design_y:.2f} µm")
print("Optimization wavelengths (nm):", cfg.wavelengths_nm())
print("Independent parameters:", cfg.parameter_shape)
print("Result directory:", OUTPUT)

## 2. Preview the geometry locally

This is the seeded initial geometry. This cell does not submit a cloud simulation.

In [ ]:
params_initial = initial_parameters(cfg)
sim_initial = build_simulation(params_initial, cfg, cfg.cases[0])
ax = sim_initial.plot_eps(z=0, monitor_alpha=0, source_alpha=0)
ax.set_title("Initial permittivity")
plt.show()

## 3. Optimize or resume

Configure your own Tidy3D account in a terminal with `tidy3d configure`. Set `RUN_OPTIMIZATION = True` when ready to submit paid jobs. `MAX_ADDITIONAL_STEPS = 10` limits this session to ten additional updates; `None` runs to the configured total.

The checkpoint preserves Adam moments, step counters, parameters and gradients. After reconfiguring your account, rerun this cell with the same output directory and settings. Do not run two optimization processes in the same output directory.

In [ ]:
RUN_OPTIMIZATION = False
MAX_ADDITIONAL_STEPS = None

if RUN_OPTIMIZATION:
    state = optimize(cfg, OUTPUT, submit=True, max_steps=MAX_ADDITIONAL_STEPS)
    print(f"Completed {state.completed}/{cfg.iterations} updates")
else:
    print("Preview mode: enable RUN_OPTIMIZATION to start or resume cloud work.")

## 4. Inspect saved progress

Each objective value belongs to the geometry before its update. The final validation evaluates the last updated geometry.

In [ ]:
checkpoint = OUTPUT / "checkpoint.npz"
if checkpoint.exists():
    with np.load(checkpoint, allow_pickle=False) as saved:
        scores = saved["scores"].copy()
        full = saved["full_params"].copy()
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
    axes[0].plot(np.arange(1, len(scores) + 1), scores)
    axes[0].set(xlabel="Update", ylabel="Objective before update")
    axes[1].imshow(full.T, origin="lower", cmap="gray", vmin=0, vmax=1,
                   extent=(0, cfg.design_x, -cfg.design_y / 2, cfg.design_y / 2))
    axes[1].set(xlabel="x (µm)", ylabel="y (µm)", title="Current design parameters")
    plt.show()
else:
    print("No optimization checkpoint yet.")

## 5. Validate the completed design

This submits additional, independent single-port simulations. Both mode and flux monitors sample 501 wavelengths. The complex array `transmission[output, input, wavelength]` comes from mode monitors. Flux powers are saved separately. Field plots use 1500 nm.

For the decoder, these are the training-orientation equal-splitting responses. Reciprocal routing and arbitrary allocation synthesis are separate tasks; see `docs/methods.md` for port and phase conventions.

In [ ]:
RUN_VALIDATION = False

if RUN_VALIDATION:
    validation_data = evaluate(cfg, OUTPUT, submit=True)
    print("Saved numerical results and figures to", OUTPUT / "validation")
else:
    print("Enable RUN_VALIDATION after completing optimization.")

## 6. Read the exported complex response

No cloud access is required to load a saved transmission archive.

In [ ]:
archive = OUTPUT / "validation" / "transmission.npz"
if archive.exists():
    with np.load(archive, allow_pickle=False) as saved:
        wavelengths_nm = saved["wavelength_nm"].copy()
        transmission = saved["transmission"].copy()
        flux_power = saved["flux_power"].copy()
    k = int(np.argmin(np.abs(wavelengths_nm - 1500)))
    print("Wavelength (nm):", wavelengths_nm[k])
    print("Amplitude |T|:\n", np.abs(transmission[:, :, k]))
    print("Phase arg(T), degrees:\n", np.angle(transmission[:, :, k], deg=True))
else:
    print("No validation archive yet.")